### Trial 1 - Text Extraction

##### raw text extraction from PDF - simple method

In [1]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import pandas as pd

# 🔧 CONFIG
PDF_PATH = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\CSI AP Aging as of 02.20.26.pdf"
OUTPUT_TEXT_FILE = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ap_aging_raw_text.txt"

# If Tesseract is not in PATH (Windows users)
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\IlaBarshilia\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"
pages = convert_from_path(PDF_PATH, dpi=300, poppler_path=r"C:\Users\IlaBarshilia\Downloads\Poppler\poppler-25.12.0\Library\bin")

all_text = []

for idx, page in enumerate(pages):
    text = pytesseract.image_to_string(page)
    all_text.append(text)

# Save raw OCR output (VERY important for debugging)
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_text))

print("✅ Raw OCR text saved:", OUTPUT_TEXT_FILE)

✅ Raw OCR text saved: C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ap_aging_raw_text.txt


##### format raw text into table format

In [2]:
import re
import pandas as pd

INPUT_FILE = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ap_aging_raw_text.txt"
OUTPUT_EXCEL = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\AP_Aging_With_Balances.xlsx"

rows = []
current_vendor_code = None
current_vendor_name = None

# ------------------------
# Regex
# ------------------------

vendor_header_pattern = re.compile(r"^([A-Z0-9]{3,10})\s+(.+?)\s+ACCT#")

amount_pattern = re.compile(r"-?\d{1,3}(?:,\d{3})*(?:[.,]\d{2})")
type_pattern = re.compile(r"\b(AP|PY|CR|DB)\b")
due_date_pattern = re.compile(r"(\d{2}/\d{2}/\d{2})\s+AP")

# ------------------------
# Helpers
# ------------------------

def normalize_number(s: str) -> float:
    if not s:
        return 0.0

    s = re.sub(r"[^0-9.,\-]", "", s)

    last_dot = s.rfind(".")
    last_comma = s.rfind(",")

    decimal_pos = max(last_dot, last_comma)

    if decimal_pos != -1:
        integer_part = re.sub(r"[.,]", "", s[:decimal_pos])
        decimal_part = s[decimal_pos + 1:]
        s = f"{integer_part}.{decimal_part}"
    else:
        s = re.sub(r"[.,]", "", s)

    return float(s)


def extract_invoice_date(line):
    cleaned = re.sub(r"^[^0-9]*", "", line)
    token = cleaned.split()[0] if cleaned else None

    if token and re.match(r"\d{2}/\d{2}/\d{2}", token):
        if token.startswith("9"):      # OCR: 91/ → 01/
            token = "0" + token[1:]
        if token.startswith("49"):     # OCR junk like 492/ → 02/
            token = "0" + token[-7:]
        return token
    return None


def extract_due_date(line):
    """
    Due Date = date token immediately before AP / PY / CR / DB.
    Apply same OCR cleanup rules as Invoice Date.
    """
    m = re.search(r"([^\s]+)\s+(AP|PY|CR|DB)\b", line)
    if not m:
        return None

    token = m.group(1)

    # Strip OCR junk before digits
    token = re.sub(r"^[^0-9]*", "", token)

    # Must look like MM/DD/YY
    if not re.match(r"\d{2}/\d{2}/\d{2}", token):
        return None

    # OCR fixes: 9x/.. or 6x/.. → 0x/..
    if token[0] in ("9", "6"):
        token = "0" + token[1:]

    # Final safety: ensure exact length MM/DD/YY
    if len(token) > 8:
        token = token[-8:]

    return token


def extract_buckets(line):
    nums = amount_pattern.findall(line)

    values = [normalize_number(n) for n in nums[1:6]]

    while len(values) < 5:
        values.append(0.0)

    return values


# ------------------------
# Parse file
# ------------------------

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for raw_line in f:
        line = raw_line.strip()
        # Normalize OCR separators but KEEP numbers
        line = re.sub(r"[=+—]+", " ", line)
        line = re.sub(r"\s+", " ", line)

        if not line:
            continue
        if line.startswith(("PROG:", "PAGE", "USER:", "SUMMARY", "TOTAL", "BALANCE")):
            continue

        vh = vendor_header_pattern.search(line)
        if vh:
            current_vendor_code = vh.group(1)
            current_vendor_name = vh.group(2).strip()
            continue

        if not current_vendor_code:
            continue

        amt = amount_pattern.search(line)
        typ = type_pattern.search(line)

        if amt and typ:
            inv_nums = re.findall(r"[A-Z0-9]{5,}", line)
            invoice_number = inv_nums[0] if inv_nums else None
            buckets = extract_buckets(line)

            rows.append({
                "Vendor Code": current_vendor_code,
                "Vendor Name": current_vendor_name,
                "Row Type": "INVOICE",
                "Invoice Date": extract_invoice_date(line),
                "Invoice Number": invoice_number,
                "Due Date": extract_due_date(line),
                "Type": typ.group(1),
                "Amount": normalize_number(amt.group()),
                "Current": buckets[0],
                "Over 30": buckets[1],
                "Over 60": buckets[2],
                "Over 90": buckets[3],
                "Over 120": buckets[4],
            })

# ------------------------
# Build DataFrame
# ------------------------

df = pd.DataFrame(rows)

df["Invoice Date"] = pd.to_datetime(
    df["Invoice Date"], format="%m/%d/%y", errors="coerce"
).dt.date

df["Due Date"] = pd.to_datetime(
    df["Due Date"], format="%m/%d/%y", errors="coerce"
).dt.date

# ------------------------
# Vendor-level BALANCE rows
# ------------------------

final_rows = []

for (vcode, vname), g in df.groupby(["Vendor Code", "Vendor Name"], sort=False):
    for _, r in g.iterrows():
        final_rows.append(r.to_dict())

    final_rows.append({
        "Vendor Code": vcode,
        "Vendor Name": vname,
        "Row Type": "BALANCE",
        "Invoice Date": None,
        "Invoice Number": None,
        "Due Date": None,
        "Type": "BALANCE",
        "Amount": g["Amount"].sum(),
        "Current": g["Current"].sum(),
        "Over 30": g["Over 30"].sum(),
        "Over 60": g["Over 60"].sum(),
        "Over 90": g["Over 90"].sum(),
        "Over 120": g["Over 120"].sum(),
    })

final_df = pd.DataFrame(final_rows)
final_df.to_excel(OUTPUT_EXCEL, index=False)

print(f"✅ Extracted {len(final_df)} rows")
print(f"✅ Output written to {OUTPUT_EXCEL}")

✅ Extracted 1015 rows
✅ Output written to C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\AP_Aging_With_Balances.xlsx


#### trial for AR PDF

In [1]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import pandas as pd

# 🔧 CONFIG
PDF_PATH = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\CSI AR Aging as of 2.19.2026.pdf"
OUTPUT_TEXT_FILE = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ar_aging_raw_text.txt"

# If Tesseract is not in PATH (Windows users)
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\IlaBarshilia\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"
pages = convert_from_path(PDF_PATH, dpi=300, poppler_path=r"C:\Users\IlaBarshilia\Downloads\Poppler\poppler-25.12.0\Library\bin")

all_text = []

for idx, page in enumerate(pages):
    text = pytesseract.image_to_string(page)
    all_text.append(text)

# Save raw OCR output (VERY important for debugging)
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_text))

print("✅ Raw OCR text saved:", OUTPUT_TEXT_FILE)

✅ Raw OCR text saved: C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ar_aging_raw_text.txt


#### Working 80% for AR PDF

In [67]:
import pytesseract
from pdf2image import convert_from_path
import pandas as pd

pages = convert_from_path(
    PDF_PATH,
    dpi=300,
    poppler_path=r"C:\Users\IlaBarshilia\Downloads\Poppler\poppler-25.12.0\Library\bin"
)

dfs = []

for page_num, page in enumerate(pages, start=1):
    df = pytesseract.image_to_data(
        page,
        output_type=pytesseract.Output.DATAFRAME
    )
    df["page"] = page_num
    dfs.append(df)

ocr_df = pd.concat(dfs, ignore_index=True)

# Keep only actual text
ocr_df = ocr_df[
    ocr_df.text.notna() &
    (ocr_df.text.str.strip() != "") &
    (ocr_df.conf != -1)
]

ocr_df.to_csv("ar_aging_ocr_with_coords.csv", index=False)

In [68]:
ROW_TOLERANCE = 10

ocr_df = ocr_df.sort_values(["page", "top", "left"]).reset_index(drop=True)

row_ids = []
current_row = 0
last_top = None

for _, r in ocr_df.iterrows():
    if last_top is None or abs(r.top - last_top) > ROW_TOLERANCE:
        current_row += 1
        last_top = r.top
    row_ids.append(current_row)

ocr_df["row_id"] = row_ids

# =========================
# CONFIG
# =========================
COLUMN_RANGES = {
    "Amount":   (900, 1100),
    "Current":  (1101, 1300),
    "Over_15":  (1301, 1500),
    "Over_30":  (1501, 1700),
    "Over_45":  (1701, 1900),
    "Over_90":  (1901, 2100),
}

import re
import pandas as pd

# =========================
# HELPERS
# =========================
def clean_number(val):
    if val is None:
        return 0.0

    val = str(val).strip()

    if val in {"", ".", "-", "--"}:
        return 0.0

    # ✅ Case 1: ONE comma, NO dot → comma is decimal separator
    if val.count(",") == 1 and "." not in val:
        val = val.replace(",", ".")

    # ✅ Otherwise: treat comma as thousands separator
    val = val.replace(",", "")

    try:
        return float(val)
    except ValueError:
        return 0.0


def detect_bucket(x):
    for bucket, (xmin, xmax) in COLUMN_RANGES.items():
        if xmin <= x <= xmax:
            return bucket
    return None


def clean_customer_name(name):
    if not name:
        return name

    # Remove leading junk
    name = re.sub(r"^[^A-Za-z0-9]+", "", name)

    # ✅ Remove trailing special characters (==, --, **, etc.)
    name = re.sub(r"[^A-Za-z0-9&.,'() ]+$", "", name)

    return name.strip()

def fix_invoice_date(date_str):
    if not date_str:
        return date_str
    date_str = date_str.strip()
    # ✅ Step 1: remove leading non-digit characters (OCR junk like Â§, =, *, etc.)
    date_str = re.sub(r"^[^\d]+", "", date_str)
    # ✅ Step 2: fix OCR error where first digit should be 0 but is 9 or 6
    if re.match(r"^[96]", date_str):
        date_str = "0" + date_str[1:]

    return date_str

def amount_mismatch_flag(row):
    """
    Returns True if Amount does NOT match any aging bucket.
    """
    amount = row["Amount"]
    buckets = ["Current", "Over_15", "Over_30", "Over_45", "Over_90"]

    return amount not in [row[b] for b in buckets if row[b] > 0]

# =========================
# CUSTOMER STATE
# =========================
current_customer_code = None
current_customer_name = None

pending_customer_code = None
pending_customer_name = None

records = []

# =========================
# MAIN LOOP (PDF ORDER PRESERVED)
# =========================
for (page, row_id), row in (
    ocr_df
    .sort_values(["page", "top", "left"])
    .groupby(["page", "row_id"])
):
    row = row.sort_values("left")
    line_text = " ".join(row.text.tolist()).strip()
    line_upper = line_text.upper()

    # --------------------------------------------------
    # BALANCE → reset pending customer buffers ONLY
    # --------------------------------------------------
    if "BALANCE" in line_upper:
        pending_customer_code = None
        pending_customer_name = None
        continue

    # --------------------------------------------------
    # CUSTOMER HEADER — CODE ONLY (e.g. WOOPRO)
    # --------------------------------------------------
    if (
        re.fullmatch(r"[A-Z0-9]{3,8}", line_text)
        and not re.search(r"\b(IN|PY|CR)\b", line_upper)
    ):
        pending_customer_code = line_text
        pending_customer_name = None
        continue

    # --------------------------------------------------
    # CUSTOMER HEADER — FULL (robust token-based)
    # --------------------------------------------------
    if "A/P" in line_upper and "REP" in line_upper and "TERMS" in line_upper:

        tokens = [str(t).strip() for t in row.text.tolist()]

        # Find customer code (first valid token)
        code = None
        for t in tokens:
            if re.fullmatch(r"[A-Z0-9]{3,8}", t):
                code = t
                break

        if code:
            # Build name from tokens between code and A/P
            name_parts = []
            seen_code = False

            for t in tokens:
                if t == code:
                    seen_code = True
                    continue
                if not seen_code:
                    continue
                if t.upper().startswith("A/P"):
                    break
                # skip OCR junk like â€, â€”
                if re.fullmatch(r"[^\w]+", t):
                    continue
                name_parts.append(t)

            current_customer_code = code
            current_customer_name = clean_customer_name(
                " ".join(name_parts)
            )

            pending_customer_code = None
            pending_customer_name = None
            continue

    # --------------------------------------------------
    # TRANSACTION ROW
    # --------------------------------------------------
    if not re.search(r"\b(IN|PY|CR)\b", line_upper):
        continue

    # Do not emit until customer is known
    if not current_customer_code:
        continue

    # --------------------------------------------------
    # EXTRACT TRANSACTION DATA
    # --------------------------------------------------
    date = row[row.text.str.contains(r"\d{1,2}/\d{1,2}/\d{2}", na=False)]
    inv  = row[row.text.str.contains(r"\d{5,}-\d+", na=False)]
    typ  = row[row.text.isin(["IN", "PY", "CR"])]

    aging = {
        "Amount": 0.0,
        "Current": 0.0,
        "Over_15": 0.0,
        "Over_30": 0.0,
        "Over_45": 0.0,
        "Over_90": 0.0,
    }

    numbers = row[row.text.astype(str).str.strip().str.contains(r"^\d[\d,]*\.?\d*$", na=False)]
    for _, n in numbers.iterrows():
        bucket = detect_bucket(n.left)
        if bucket:
            value = clean_number(n.text)
            if value >= 10:                    # 🔑 FILTER OCR NOISE
                aging[bucket] = value

    raw_date = date.text.iloc[0] if not date.empty else None

    records.append({
        "Customer Code": current_customer_code,
        "Customer Name": current_customer_name,
        "Invoice Date": fix_invoice_date(raw_date),
        "Invoice Number": inv.text.iloc[0] if not inv.empty else None,
        "Type": typ.text.iloc[0] if not typ.empty else None,
        **aging
    })

# =========================
# FINAL OUTPUT
# =========================
final_df = pd.DataFrame(records)

print("✅ Structured AR Aging extracted")

# =========================
# INSERT BALANCE ROW AFTER EACH CUSTOMER BLOCK
# =========================

financial_cols = ["Amount", "Current", "Over_15", "Over_30", "Over_45", "Over_90"]
output_rows = []

current_customer = None
running_balance = {col: 0.0 for col in financial_cols}

for _, row in final_df.iterrows():
    cust_key = (row["Customer Code"], row["Customer Name"])

    # Customer change detected → insert BALANCE for previous customer
    if current_customer and cust_key != current_customer:
        balance_row = {
            "Customer Code": current_customer[0],
            "Customer Name": current_customer[1],
            "Invoice Date": None,
            "Invoice Number": None,
            "Type": "BALANCE",
            **running_balance}
        output_rows.append(balance_row)

        # reset running totals
        running_balance = {col: 0.0 for col in financial_cols}

    # Track current customer
    current_customer = cust_key

    # Accumulate amounts (skip balance rows just in case)
    if row["Type"] != "BALANCE":
        for col in financial_cols:
            running_balance[col] += row[col]

    # Append original row
    output_rows.append(row.to_dict())

# Insert BALANCE for the final customer
if current_customer:
    balance_row = {
        "Customer Code": current_customer[0],
        "Customer Name": current_customer[1],
        "Invoice Date": None,
        "Invoice Number": None,
        "Type": "BALANCE",
        **running_balance
    }
    output_rows.append(balance_row)

# Final dataframe with in-order balance rows
final_df = pd.DataFrame(output_rows)
final_df["Amount_Mismatch"] = final_df.apply(
    lambda r: amount_mismatch_flag(r) if r["Type"] != "BALANCE" else False,
    axis=1
)
print(final_df.shape)

✅ Structured AR Aging extracted
(1038, 12)


In [69]:
trial = final_df[final_df['Amount_Mismatch']==True]
trial.shape

(381, 12)

In [70]:
import pandas as pd

output_excel = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ar_aging_parsed.xlsx"

with pd.ExcelWriter(output_excel, engine="xlsxwriter") as writer:
    final_df.to_excel(writer, index=False, sheet_name="AR_Aging")

    workbook  = writer.book
    worksheet = writer.sheets["AR_Aging"]

    red_format = workbook.add_format({
        "bg_color": "#FFC7CE",   # light red
        "font_color": "#9C0006"})

    mismatch_col_idx = final_df.columns.get_loc("Amount_Mismatch")

    # Highlight entire row when mismatch is TRUE
    worksheet.conditional_format(
        1, 0,
        len(final_df), len(final_df.columns) - 1,
        {   "type": "formula",
            "criteria": f"=${chr(65 + mismatch_col_idx)}2=TRUE",
            "format": red_format})

print(f"✅ Excel file written with highlights: {output_excel}")

✅ Excel file written with highlights: C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ar_aging_parsed.xlsx


#### trial for correct customer code

In [65]:
import pytesseract
from pdf2image import convert_from_path
import pandas as pd
import cv2
import numpy as np
from PIL import Image

custom_config = r"""
--oem 3
--psm 6
-c tessedit_char_whitelist=0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz.,-/:()
"""

def preprocess_image(pil_img):
    img = np.array(pil_img)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return Image.fromarray(gray)

pages = convert_from_path(
    PDF_PATH,
    dpi=350,
    poppler_path=r"C:\Users\IlaBarshilia\Downloads\Poppler\poppler-25.12.0\Library\bin"
)

dfs = []

for page_num, page in enumerate(pages, start=1):
    page = preprocess_image(page)
    df = pytesseract.image_to_data(
        page,
        config="--oem 3 --psm 3",
        output_type=pytesseract.Output.DATAFRAME
    )
    df["page"] = page_num
    dfs.append(df)

ocr_df = pd.concat(dfs, ignore_index=True)

ocr_df = ocr_df[
    ocr_df.text.notna() &
    (ocr_df.text.str.strip() != "") &
    (ocr_df.conf >= 30)]

ocr_df.to_csv(r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ar_aging_ocr_with_coords_new.csv", index=False)

ROW_TOLERANCE = 10

ocr_df = ocr_df.sort_values(["page", "top", "left"]).reset_index(drop=True)

row_ids = []
current_row = 0
last_top = None

for _, r in ocr_df.iterrows():
    if last_top is None or abs(r.top - last_top) > ROW_TOLERANCE:
        current_row += 1
        last_top = r.top
    row_ids.append(current_row)

ocr_df["row_id"] = row_ids

# =========================
# CONFIG
# =========================
COLUMN_RANGES = {
    "Amount":   (900, 1100),
    "Current":  (1101, 1300),
    "Over_15":  (1301, 1500),
    "Over_30":  (1501, 1700),
    "Over_45":  (1701, 1900),
    "Over_90":  (1901, 2100),
}

import re
import pandas as pd

# =========================
# HELPERS
# =========================
def clean_number(val):
    if val is None:
        return 0.0

    val = str(val).strip()

    if val in {"", ".", "-", "--"}:
        return 0.0

    # ✅ Case 1: ONE comma, NO dot → comma is decimal separator
    if val.count(",") == 1 and "." not in val:
        val = val.replace(",", ".")

    # ✅ Otherwise: treat comma as thousands separator
    val = val.replace(",", "")

    try:
        return float(val)
    except ValueError:
        return 0.0


def detect_bucket(x):
    for bucket, (xmin, xmax) in COLUMN_RANGES.items():
        if xmin <= x <= xmax:
            return bucket
    return None


def clean_customer_name(name):
    if not name:
        return name

    # Remove leading junk
    name = re.sub(r"^[^A-Za-z0-9]+", "", name)

    # ✅ Remove trailing special characters (==, --, **, etc.)
    name = re.sub(r"[^A-Za-z0-9&.,'() ]+$", "", name)

    return name.strip()

def fix_invoice_date(date_str):
    if not date_str:
        return date_str
    date_str = date_str.strip()
    # ✅ Step 1: remove leading non-digit characters (OCR junk like Â§, =, *, etc.)
    date_str = re.sub(r"^[^\d]+", "", date_str)
    # ✅ Step 2: fix OCR error where first digit should be 0 but is 9 or 6
    if re.match(r"^[96]", date_str):
        date_str = "0" + date_str[1:]

    return date_str

def amount_mismatch_flag(row):
    """
    Returns True if Amount does NOT match any aging bucket.
    """
    amount = row["Amount"]
    buckets = ["Current", "Over_15", "Over_30", "Over_45", "Over_90"]

    return amount not in [row[b] for b in buckets if row[b] > 0]

# =========================
# CUSTOMER STATE
# =========================
current_customer_code = None
current_customer_name = None

pending_customer_code = None
pending_customer_name = None

records = []

# =========================
# MAIN LOOP (PDF ORDER PRESERVED)
# =========================
for (page, row_id), row in (
    ocr_df
    .sort_values(["page", "top", "left"])
    .groupby(["page", "row_id"])
):
    row = row.sort_values("left")
    line_text = " ".join(row.text.tolist()).strip()
    line_upper = line_text.upper()

    # --------------------------------------------------
    # BALANCE → reset pending customer buffers ONLY
    # --------------------------------------------------
    if "BALANCE" in line_upper:
        pending_customer_code = None
        pending_customer_name = None
        continue

    # --------------------------------------------------
    # CUSTOMER HEADER — CODE ONLY (e.g. WOOPRO)
    # --------------------------------------------------
    if (
        re.fullmatch(r"[A-Z0-9]{3,8}", line_text)
        and not re.search(r"\b(IN|PY|CR)\b", line_upper)
    ):
        pending_customer_code = line_text
        pending_customer_name = None
        continue
    # --------------------------------------------------
    # CUSTOMER HEADER — FULL (robust token-based)
    # --------------------------------------------------
    if "A/P" in line_upper and "REP" in line_upper and "TERMS" in line_upper:

        tokens = [str(t).strip() for t in row.text.tolist()]

        # Find customer code (first valid token)
        code = None
        for t in tokens:
            if re.fullmatch(r"[A-Z0-9]{3,8}", t):
                code = t
                break

        if code:
            # Build name from tokens between code and A/P
            name_parts = []
            seen_code = False

            for t in tokens:
                if t == code:
                    seen_code = True
                    continue
                if not seen_code:
                    continue
                if t.upper().startswith("A/P"):
                    break
                # skip OCR junk like â€, â€”
                if not re.search(r"[A-Z0-9]", t):
                    continue

                name_parts.append(t)

            current_customer_code = code
            current_customer_name = clean_customer_name(
                " ".join(name_parts)
            )

            pending_customer_code = None
            pending_customer_name = None
            continue

    # --------------------------------------------------
    # TRANSACTION ROW
    # --------------------------------------------------
    if not re.search(r"\b(IN|PY|CR)\b", line_upper):
        continue

    # Do not emit until customer is known
    if not current_customer_code:
        continue

    # --------------------------------------------------
    # EXTRACT TRANSACTION DATA
    # --------------------------------------------------
    date = row[row.text.str.contains(r"\d{1,2}/\d{1,2}/\d{2}", na=False)]
    inv  = row[row.text.str.contains(r"\d{5,}-\d+", na=False)]
    typ  = row[row.text.isin(["IN", "PY", "CR"])]

    aging = {
        "Amount": 0.0,
        "Current": 0.0,
        "Over_15": 0.0,
        "Over_30": 0.0,
        "Over_45": 0.0,
        "Over_90": 0.0,
    }

    numbers = row[row.text.astype(str).str.strip().str.contains(r"^\d[\d,]*\.?\d*$", na=False)]
    for _, n in numbers.iterrows():
        bucket = detect_bucket(n.left)
        if bucket:
            value = clean_number(n.text)
            if value >= 10:                    # 🔑 FILTER OCR NOISE
                aging[bucket] = value

    raw_date = date.text.iloc[0] if not date.empty else None

    records.append({
        "Customer Code": current_customer_code,
        "Customer Name": current_customer_name,
        "Invoice Date": fix_invoice_date(raw_date),
        "Invoice Number": inv.text.iloc[0] if not inv.empty else None,
        "Type": typ.text.iloc[0] if not typ.empty else None,
        **aging
    })

# =========================
# FINAL OUTPUT
# =========================
final_df = pd.DataFrame(records)

print("✅ Structured AR Aging extracted")

# =========================
# INSERT BALANCE ROW AFTER EACH CUSTOMER BLOCK
# =========================

financial_cols = ["Amount", "Current", "Over_15", "Over_30", "Over_45", "Over_90"]
output_rows = []

current_customer = None
running_balance = {col: 0.0 for col in financial_cols}

for _, row in final_df.iterrows():
    cust_key = (row["Customer Code"], row["Customer Name"])

    # Customer change detected → insert BALANCE for previous customer
    if current_customer and cust_key != current_customer:
        balance_row = {
            "Customer Code": current_customer[0],
            "Customer Name": current_customer[1],
            "Invoice Date": None,
            "Invoice Number": None,
            "Type": "BALANCE",
            **running_balance}
        output_rows.append(balance_row)

        # reset running totals
        running_balance = {col: 0.0 for col in financial_cols}

    # Track current customer
    current_customer = cust_key

    # Accumulate amounts (skip balance rows just in case)
    if row["Type"] != "BALANCE":
        for col in financial_cols:
            running_balance[col] += row[col]

    # Append original row
    output_rows.append(row.to_dict())

# Insert BALANCE for the final customer
if current_customer:
    balance_row = {
        "Customer Code": current_customer[0],
        "Customer Name": current_customer[1],
        "Invoice Date": None,
        "Invoice Number": None,
        "Type": "BALANCE",
        **running_balance
    }
    output_rows.append(balance_row)

# Final dataframe with in-order balance rows
final_df = pd.DataFrame(output_rows)
final_df["Amount_Mismatch"] = final_df.apply(
    lambda r: amount_mismatch_flag(r) if r["Type"] != "BALANCE" else False,
    axis=1
)
print(final_df.shape)
final_df.head(20)

trial = final_df[final_df['Amount_Mismatch']==True]
print(trial.shape)

import pandas as pd

output_excel = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ar_aging_parsed.xlsx"

with pd.ExcelWriter(output_excel, engine="xlsxwriter") as writer:
    final_df.to_excel(writer, index=False, sheet_name="AR_Aging")

    workbook  = writer.book
    worksheet = writer.sheets["AR_Aging"]

    red_format = workbook.add_format({
        "bg_color": "#FFC7CE",   # light red
        "font_color": "#9C0006"})

    mismatch_col_idx = final_df.columns.get_loc("Amount_Mismatch")

    # Highlight entire row when mismatch is TRUE
    worksheet.conditional_format(
        1, 0,
        len(final_df), len(final_df.columns) - 1,
        {   "type": "formula",
            "criteria": f"=${chr(65 + mismatch_col_idx)}2=TRUE",
            "format": red_format})

print(f"✅ Excel file written with highlights: {output_excel}")

✅ Structured AR Aging extracted
(859, 12)
(561, 12)
✅ Excel file written with highlights: C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ar_aging_parsed.xlsx


#### Next Code

In [62]:
import re
import pandas as pd
from pathlib import Path

# ---------- CONFIG ----------
INPUT_FILE = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ar_aging_raw_text.txt"
OUTPUT_FILE = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ar_aging_parsed.csv"

# ---------- REGEX PATTERNS ----------
customer_header_re = re.compile(
    r"^([A-Z0-9]{3,8})\s*=\s*(.+?)(?:A/P:|REP:|TERMS:|$)",
    re.IGNORECASE
)

invoice_line_re = re.compile(
    r"""
    ^\*{0,2}\s*
    (?P<date>\d{1,2}/\d{1,2}/\d{2})\s+
    (?P<invoice>\d{5,}-\d+|\d{5,})\s+
    (?P<type>IN|PY|CR)\s+
    (?P<amount>-?[\d,.]+)
    (?:\s+(?P<current>-?[\d,.]+))?
    """,
    re.VERBOSE
)

balance_re = re.compile(r"^BALANCE", re.IGNORECASE)

# ---------- HELPERS ----------
def to_float(val):
    """
    Safely convert messy PDF-extracted numeric strings to float.
    Handles None, '.', empty strings, commas, and OCR noise.
    """
    if val is None:
        return 0.0

    val = val.strip()

    # common garbage values from PDF extraction
    if val in {"", ".", "-", "--"}:
        return 0.0

    # fix cases like '896. 00' → '896.00'
    val = val.replace(" ", "").replace(",", "")

    try:
        return float(val)
    except ValueError:
        return 0.0

# ---------- MAIN PARSER ----------
rows = []
current_customer_code = None
current_customer_name = None

with open(INPUT_FILE, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        # --- CUSTOMER HEADER ---
        cust_match = customer_header_re.search(line)
        if cust_match:
            current_customer_code = cust_match.group(1).strip()
            current_customer_name = cust_match.group(2).strip()
            continue

        # Skip balance lines
        if balance_re.match(line):
            continue

        # --- INVOICE LINE ---
        inv_match = invoice_line_re.search(line)
        if inv_match and current_customer_code:
            amount = to_float(inv_match.group("amount"))
            current_amt = to_float(inv_match.group("current"))

            # Aging logic:
            # Default everything to 0
            aging = {
                "Current": 0.0,
                "Over_15": 0.0,
                "Over_30": 0.0,
                "Over_45": 0.0,
                "Over_90": 0.0,
            }

            # If CURRENT column exists → current bucket
            if current_amt != 0:
                aging["Current"] = current_amt
            else:
                # otherwise treat entire amount as aged
                aging["Over_30"] = amount  # safe default

            rows.append({
                "Customer Code": current_customer_code,
                "Customer Name": current_customer_name,
                "Invoice Date": inv_match.group("date"),
                "Invoice Number": inv_match.group("invoice"),
                "Type": inv_match.group("type"),
                "Invoice Amount": amount,
                **aging
            })

# ---------- OUTPUT ----------
df = pd.DataFrame(rows)

df.to_csv(OUTPUT_FILE, index=False)

print(f"✅ Parsed {len(df)} invoice rows")
print(f"✅ Output written to {OUTPUT_FILE}")

✅ Parsed 649 invoice rows
✅ Output written to C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ar_aging_parsed.csv


#### trial 2 - advanced raw text parsing from PDF

In [ ]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import pandas as pd

# 🔧 CONFIG
PDF_PATH = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\CSI AP Aging as of 02.20.26.pdf"
OUTPUT_TEXT_FILE = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ap_aging_raw_text.txt"
OUTPUT_EXCEL = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\CSI_AP_Aging_OCR.xlsx"

tesseract_config = (
    "--oem 3 "                 # LSTM OCR engine
    "--psm 4 "                 # Assume uniform blocks of text (tables!)
    "-c preserve_interword_spaces=1 "
    "-c tessedit_char_whitelist=0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz./-:, ")

# If Tesseract is not in PATH (Windows users)
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\IlaBarshilia\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"
pages = convert_from_path(PDF_PATH, dpi=300, poppler_path=r"C:\Users\IlaBarshilia\Downloads\Poppler\poppler-25.12.0\Library\bin")

def preprocess_image(img: Image.Image) -> Image.Image:
    img = img.convert("L")          # grayscale only
    img = img.resize(
        (img.width * 2, img.height * 2),
        Image.BICUBIC
    )
    return img

all_text = []

for idx, page in enumerate(pages):
    clean_page = preprocess_image(page)

    text = pytesseract.image_to_string(
        clean_page,
        config=tesseract_config)

    all_text.append(text)


# Save raw OCR output (VERY important for debugging)
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_text))

print("✅ Raw OCR text saved:", OUTPUT_TEXT_FILE)

✅ Raw OCR text saved: C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ap_aging_raw_text.txt


In [ ]:
import re
import pandas as pd

INPUT_FILE = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ap_aging_raw_text.txt"
OUTPUT_EXCEL = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\AP_Aging_With_Balances.xlsx"

rows = []
current_vendor_code = None
current_vendor_name = None

# ------------------------
# Regex
# ------------------------

vendor_header_pattern = re.compile(
    r"^([A-Z0-9]{3,10})\s+(.+?)\s+ACCT#"
)

amount_pattern = re.compile(r"-?\d{1,3}(?:,\d{3})*(?:[.,]\d{2})")
type_pattern = re.compile(r"\b(AP|PY|CR|DB)\b")
due_date_pattern = re.compile(r"(\d{2}/\d{2}/\d{2})\s+AP")

# ------------------------
# Helpers
# ------------------------

def normalize_number(s: str) -> float:
    if not s:
        return 0.0

    s = re.sub(r"[^0-9.,\-]", "", s)

    last_dot = s.rfind(".")
    last_comma = s.rfind(",")

    decimal_pos = max(last_dot, last_comma)

    if decimal_pos != -1:
        integer_part = re.sub(r"[.,]", "", s[:decimal_pos])
        decimal_part = s[decimal_pos + 1:]
        s = f"{integer_part}.{decimal_part}"
    else:
        s = re.sub(r"[.,]", "", s)

    return float(s)


def extract_invoice_date(line):
    cleaned = re.sub(r"^[^0-9]*", "", line)
    token = cleaned.split()[0] if cleaned else None

    if token and re.match(r"\d{2}/\d{2}/\d{2}", token):
        if token.startswith("9"):      # OCR: 91/ → 01/
            token = "0" + token[1:]
        if token.startswith("49"):     # OCR junk like 492/ → 02/
            token = "0" + token[-7:]
        return token
    return None


def extract_due_date(line):
    """
    Due Date = date token immediately before AP / PY / CR / DB.
    Apply same OCR cleanup rules as Invoice Date.
    """
    m = re.search(r"([^\s]+)\s+(AP|PY|CR|DB)\b", line)
    if not m:
        return None

    token = m.group(1)

    # Strip OCR junk before digits
    token = re.sub(r"^[^0-9]*", "", token)

    # Must look like MM/DD/YY
    if not re.match(r"\d{2}/\d{2}/\d{2}", token):
        return None

    # OCR fixes: 9x/.. or 6x/.. → 0x/..
    if token[0] in ("9", "6"):
        token = "0" + token[1:]

    # Final safety: ensure exact length MM/DD/YY
    if len(token) > 8:
        token = token[-8:]

    return token


def extract_buckets(line):
    nums = amount_pattern.findall(line)

    values = [normalize_number(n) for n in nums[1:6]]

    while len(values) < 5:
        values.append(0.0)

    return values


# ------------------------
# Parse file
# ------------------------

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for raw_line in f:
        line = raw_line.strip()
        # Normalize OCR separators but KEEP numbers
        line = re.sub(r"[=+—]+", " ", line)
        line = re.sub(r"\s+", " ", line)

        if not line:
            continue
        if line.startswith(("PROG:", "PAGE", "USER:", "SUMMARY", "TOTAL", "BALANCE")):
            continue

        vh = vendor_header_pattern.search(line)
        if vh:
            current_vendor_code = vh.group(1)
            current_vendor_name = vh.group(2).strip()
            continue

        if not current_vendor_code:
            continue

        amt = amount_pattern.search(line)
        typ = type_pattern.search(line)

        if amt and typ:
            inv_nums = re.findall(r"[A-Z0-9]{5,}", line)
            invoice_number = inv_nums[0] if inv_nums else None
            buckets = extract_buckets(line)

            rows.append({
                "Vendor Code": current_vendor_code,
                "Vendor Name": current_vendor_name,
                "Row Type": "INVOICE",
                "Invoice Date": extract_invoice_date(line),
                "Invoice Number": invoice_number,
                "Due Date": extract_due_date(line),
                "Type": typ.group(1),
                "Amount": normalize_number(amt.group()),
                "Current": buckets[0],
                "Over 30": buckets[1],
                "Over 60": buckets[2],
                "Over 90": buckets[3],
                "Over 120": buckets[4],
            })

# ------------------------
# Build DataFrame
# ------------------------

df = pd.DataFrame(rows)

df["Invoice Date"] = pd.to_datetime(
    df["Invoice Date"], format="%m/%d/%y", errors="coerce"
).dt.date

df["Due Date"] = pd.to_datetime(
    df["Due Date"], format="%m/%d/%y", errors="coerce"
).dt.date

# ------------------------
# Vendor-level BALANCE rows
# ------------------------

final_rows = []

for (vcode, vname), g in df.groupby(["Vendor Code", "Vendor Name"], sort=False):
    for _, r in g.iterrows():
        final_rows.append(r.to_dict())

    final_rows.append({
        "Vendor Code": vcode,
        "Vendor Name": vname,
        "Row Type": "BALANCE",
        "Invoice Date": None,
        "Invoice Number": None,
        "Due Date": None,
        "Type": "BALANCE",
        "Amount": g["Amount"].sum(),
        "Current": g["Current"].sum(),
        "Over 30": g["Over 30"].sum(),
        "Over 60": g["Over 60"].sum(),
        "Over 90": g["Over 90"].sum(),
        "Over 120": g["Over 120"].sum(),
    })

final_df = pd.DataFrame(final_rows)
final_df.to_excel(OUTPUT_EXCEL, index=False)

print(f"✅ Extracted {len(final_df)} rows")
print(f"✅ Output written to {OUTPUT_EXCEL}")

#### trial 3 combination of codes to extract raw text

In [22]:
import re
import pytesseract
from pdf2image import convert_from_path
from PIL import Image

# -------------------------------------------------
# 🔧 CONFIG
# -------------------------------------------------

PDF_PATH = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\CSI AP Aging as of 02.20.26.pdf"
OUTPUT_TEXT_FILE = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ap_aging_raw_text.txt"

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\IlaBarshilia\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"
)

POPPLER_PATH = r"C:\Users\IlaBarshilia\Downloads\Poppler\poppler-25.12.0\Library\bin"

# -------------------------------------------------
# 📄 Convert PDF to images
# -------------------------------------------------

pages = convert_from_path(
    PDF_PATH,
    dpi=300,
    poppler_path=POPPLER_PATH
)

# -------------------------------------------------
# ⚙️ Tesseract config (spacing-friendly)
# -------------------------------------------------

tesseract_config = (
    "--oem 3 "
    "--psm 4 "  # ✅ preserves spacing much better than psm 6
    "-c preserve_interword_spaces=1 "
    "-c tessedit_char_whitelist="
    "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz./-:, "
)

# -------------------------------------------------
# 🖼️ Image preprocessing (gentle, spacing-safe)
# -------------------------------------------------

def preprocess_image(img: Image.Image) -> Image.Image:
    img = img.convert("L")  # grayscale only
    img = img.resize(
        (img.width * 2, img.height * 2),
        Image.BICUBIC
    )
    return img

# -------------------------------------------------
# 🔍 OCR passes
# -------------------------------------------------

def ocr_default(img: Image.Image) -> str:
    return pytesseract.image_to_string(img)

def ocr_clean(img: Image.Image) -> str:
    clean = preprocess_image(img)
    return pytesseract.image_to_string(clean, config=tesseract_config)

# -------------------------------------------------
# 🧠 Line scoring & merge logic
# -------------------------------------------------

def score_line(line: str) -> int:
    score = 0
    if re.search(r"\d{2}/\d{2}/\d{2}", line):
        score += 2
    if re.search(r"\b(AP|PY|CR|DB)\b", line):
        score += 2
    score += len(re.findall(r"-?\d+[.,]\d{2}", line))
    score -= len(re.findall(r"[£¥§�]", line))
    return score

def merge_ocr_text(raw_text: str, clean_text: str) -> str:
    raw_lines = raw_text.splitlines()
    clean_lines = clean_text.splitlines()

    merged = []
    max_len = max(len(raw_lines), len(clean_lines))

    for i in range(max_len):
        raw = raw_lines[i] if i < len(raw_lines) else ""
        clean = clean_lines[i] if i < len(clean_lines) else ""

        if score_line(clean) >= score_line(raw):
            merged.append(clean)
        else:
            merged.append(raw)

    return "\n".join(merged)

# -------------------------------------------------
# 🚀 Run hybrid OCR
# -------------------------------------------------

all_text = []

for idx, page in enumerate(pages, start=1):
    raw_text = ocr_default(page)
    clean_text = ocr_clean(page)
    final_text = merge_ocr_text(raw_text, clean_text)
    all_text.append(final_text)

# -------------------------------------------------
# 💾 Save output
# -------------------------------------------------

with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_text))

print("✅ Raw OCR text saved:")
print(OUTPUT_TEXT_FILE)

🔍 OCR page 1/55
🔍 OCR page 2/55
🔍 OCR page 3/55
🔍 OCR page 4/55
🔍 OCR page 5/55
🔍 OCR page 6/55
🔍 OCR page 7/55
🔍 OCR page 8/55
🔍 OCR page 9/55
🔍 OCR page 10/55
🔍 OCR page 11/55
🔍 OCR page 12/55
🔍 OCR page 13/55
🔍 OCR page 14/55
🔍 OCR page 15/55
🔍 OCR page 16/55
🔍 OCR page 17/55
🔍 OCR page 18/55
🔍 OCR page 19/55
🔍 OCR page 20/55
🔍 OCR page 21/55
🔍 OCR page 22/55
🔍 OCR page 23/55
🔍 OCR page 24/55
🔍 OCR page 25/55
🔍 OCR page 26/55
🔍 OCR page 27/55
🔍 OCR page 28/55
🔍 OCR page 29/55
🔍 OCR page 30/55
🔍 OCR page 31/55
🔍 OCR page 32/55
🔍 OCR page 33/55
🔍 OCR page 34/55
🔍 OCR page 35/55
🔍 OCR page 36/55
🔍 OCR page 37/55
🔍 OCR page 38/55
🔍 OCR page 39/55
🔍 OCR page 40/55
🔍 OCR page 41/55
🔍 OCR page 42/55
🔍 OCR page 43/55
🔍 OCR page 44/55
🔍 OCR page 45/55
🔍 OCR page 46/55
🔍 OCR page 47/55
🔍 OCR page 48/55
🔍 OCR page 49/55
🔍 OCR page 50/55
🔍 OCR page 51/55
🔍 OCR page 52/55
🔍 OCR page 53/55
🔍 OCR page 54/55
🔍 OCR page 55/55
✅ Raw OCR text saved:
C:\Users\IlaBarshilia\Downloads\PDF Text Parser at O

#### Combine Data from multiple PDFs in a single raw file for further processing

In [ ]:
import os
import pytesseract
from pdf2image import convert_from_path
import pandas as pd

# 🔧 CONFIG
PDF_FOLDER = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA"
OUTPUT_TEXT_FILE = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\all_pdfs_raw_text.txt"
OUTPUT_CSV_FILE = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\ocr_output.csv"

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\IlaBarshilia\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"
)

POPPLER_PATH = r"C:\Users\IlaBarshilia\Downloads\Poppler\poppler-25.12.0\Library\bin"

all_records = []

# ✅ Loop through every PDF in folder
for file_name in os.listdir(PDF_FOLDER):
    if file_name.lower().endswith(".pdf"):
        pdf_path = os.path.join(PDF_FOLDER, file_name)
        print(f"📄 Processing: {file_name}")

        pages = convert_from_path(
            pdf_path,
            dpi=300,
            poppler_path=POPPLER_PATH
        )

        for page_num, page in enumerate(pages, start=1):
            text = pytesseract.image_to_string(page)

            all_records.append({
                "pdf_name": file_name,
                "page_number": page_num,
                "text": text
            })

# ✅ Convert to DataFrame
df_ocr = pd.DataFrame(all_records)

# ✅ Save structured output (BEST for analysis)
df_ocr.to_csv(OUTPUT_CSV_FILE, index=False, encoding="utf-8")

# ✅ Save raw combined text (debug-friendly)
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    for record in all_records:
        f.write(
            f"\n\n==== PDF: {record['pdf_name']} | Page: {record['page_number']} ====\n"
        )
        f.write(record["text"])

print("✅ OCR complete")
print("📄 Text file:", OUTPUT_TEXT_FILE)
print("📊 CSV file:", OUTPUT_CSV_FILE)

#### Another PDF Data Extraction - CSI Trial balance at MAR31 2026

In [42]:
import pytesseract
from pdf2image import convert_from_path
import pandas as pd

PDF_PATH = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\CSI Trial balance at MAR31 2026.pdf"

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\IlaBarshilia\AppData\Local\Programs\Tesseract-OCR\tesseract.exe")

pages = convert_from_path(
    PDF_PATH,
    dpi=300,
    poppler_path=r"C:\Users\IlaBarshilia\Downloads\Poppler\poppler-25.12.0\Library\bin")

all_rows = []

for page_no, page in enumerate(pages, start=1):
    data = pytesseract.image_to_data(
        page,
        output_type=pytesseract.Output.DATAFRAME,
        config="--psm 6")

    data["page"] = page_no
    all_rows.append(data)

ocr_df = pd.concat(all_rows, ignore_index=True)

# Keep only real text
ocr_df = ocr_df[
    (ocr_df.text.notna()) &
    (ocr_df.text.str.strip() != "")]


row_keys = ["page", "block_num", "par_num", "line_num"]

ocr_df["row_uid"] = (
    ocr_df[row_keys]
    .astype(str)
    .agg("_".join, axis=1)
)

ocr_df["x_center"] = ocr_df["left"] + (ocr_df["width"] / 2)
ocr_df

,level,page_num,block_num,par_num,line_num,word_num,left,top,width,height,conf,text,page,row_uid,x_center
4,5,1,1,1,1,1,103,102,29,36,32.384888,"""",1,1_1_1_1,117.5
5,5,1,1,1,1,2,1097,149,84,35,34.966583,CARGO,1,1_1_1_1,1139.0
6,5,1,1,1,1,3,1202,149,136,35,34.966583,SERVICES.,1,1_1_1_1,1270.0
7,5,1,1,1,1,4,1359,149,49,35,72.134384,INC,1,1_1_1_1,1383.5
8,5,1,1,1,1,5,2159,147,66,35,79.706123,PAGE,1,1_1_1_1,2192.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4493,5,1,1,1,5,4,1561,478,44,41,8.923569,"23,",15,15_1_1_5,1583.0
4494,5,1,1,1,5,5,1613,478,171,41,8.923569,"706,401.93",15,15_1_1_5,1698.5
4495,5,1,1,1,5,6,1857,477,220,42,4.669983,"23,610,235.40",15,15_1_1_5,1967.0
4496,5,1,1,1,5,7,2154,477,240,41,27.209305,"-40,011,478.",15,15_1_1_5,2274.0


In [43]:
COLS = {
    "account_id":    (80,  250),     # leftmost numeric IDs
    "account_name":  (250, 1100),    # long text block
    "account_type":  (1100, 1250),   # A / L / E / I / C
    "category_dept": (1250, 1450),   # IND / CHI
    "beg_balance":   (1450, 1750),
    "debits":        (1750, 2050),
    "credits":       (2050, 2300),
    "balance":       (2300, 2600),
}

def assign_column(x):
    for col, (x1, x2) in COLS.items():
        if x1 <= x <= x2:
            return col
    return None

ocr_df["column"] = ocr_df["x_center"].apply(assign_column)
ocr_df

,level,page_num,block_num,par_num,line_num,word_num,left,top,width,height,conf,text,page,row_uid,x_center,column
4,5,1,1,1,1,1,103,102,29,36,32.384888,"""",1,1_1_1_1,117.5,account_id
5,5,1,1,1,1,2,1097,149,84,35,34.966583,CARGO,1,1_1_1_1,1139.0,account_type
6,5,1,1,1,1,3,1202,149,136,35,34.966583,SERVICES.,1,1_1_1_1,1270.0,category_dept
7,5,1,1,1,1,4,1359,149,49,35,72.134384,INC,1,1_1_1_1,1383.5,category_dept
8,5,1,1,1,1,5,2159,147,66,35,79.706123,PAGE,1,1_1_1_1,2192.0,credits
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4493,5,1,1,1,5,4,1561,478,44,41,8.923569,"23,",15,15_1_1_5,1583.0,beg_balance
4494,5,1,1,1,5,5,1613,478,171,41,8.923569,"706,401.93",15,15_1_1_5,1698.5,beg_balance
4495,5,1,1,1,5,6,1857,477,220,42,4.669983,"23,610,235.40",15,15_1_1_5,1967.0,debits
4496,5,1,1,1,5,7,2154,477,240,41,27.209305,"-40,011,478.",15,15_1_1_5,2274.0,credits


In [44]:
ocr_df.to_excel('Check_trial_data.xlsx')

In [ ]:


def has_any_number(row):
        return any(row[c] for c in ["beg_balance", "debits", "credits", "balance"])

rows = []

for row_uid, grp in ocr_df.groupby("row_uid"):
    row = {c: [] for c in COLS}

    for _, r in grp.iterrows():
        if r.column:
            row[r.column].append(r.text)

    final = {k: " ".join(v).strip() for k, v in row.items()}

    if has_any_number(final):
        rows.append(final)

df = pd.DataFrame(rows)
df

,account_id,account_name,account_type,category_dept,beg_balance,debits,credits,balance
0,,,CARGO,SERVICES INC,,,"PAGE NO,",10
1,,,G/L TRIAL,BALANCE,,,04/09/26,
2,ACCOUNT,ID NAME ACCOUNT TYPE,CATEGORY/DEPT,"BEG, BALANCE",DEBITS,CREDITS,BALANCE,
3,4119¢,CHI INLAND FREIGHT 0 CHI,,"830,53",0.00,9.00,,830.53
4,4165,MISCELLANEQUS-ATR 0 “IND,,"25, 235,624.49","42,265.94",14.80,"= 25,277,875.63",|
...,...,...,...,...,...,...,...,...
393,4113,AIRLINE DELIVERY 0 ND,,"348,065.67",9.00,0.00,,"348,065.67"
394,4113¢,CHI AIRLINE DELIVERY 0 CHI,,"8,849.39",0.09,0.00,,"8,049,390"
395,4116,——_TNSURANCE 0 IND,,"9,543.65",0.09,9.00,,"0,543.65"
396,4116C,CHI TNSURANCE 0 cH,,420.21,0.09,9.00,,0.0


In [51]:
import pytesseract
import pandas as pd
from pdf2image import convert_from_path
import re

PDF_PATH = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\CSI Trial balance at MAR31 2026.pdf"
OUTPUT = r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\CSI_Trial_Balance_MAR31_2026.xlsx"

images = convert_from_path(PDF_PATH, dpi=300,poppler_path=r"C:\Users\IlaBarshilia\Downloads\Poppler\poppler-25.12.0\Library\bin")

rows = []

def clean_number(x):
    x = x.replace(",", "")
    x = x.replace("©", "0")
    x = x.replace("=", "")
    try:
        return float(x)
    except:
        return None

for page_no, img in enumerate(images, start=1):

    data = pytesseract.image_to_data(
        img,
        config="--psm 6",
        output_type=pytesseract.Output.DATAFRAME
    )

    data = data.dropna(subset=["text"])
    data = data[data.text.str.strip() != ""]

    # Group words by line using Y position
    data["y_group"] = (data.top / 20).round().astype(int)

    for _, group in data.groupby("y_group"):

        words = group.sort_values("left")["text"].tolist()

        # Collect numeric-looking tokens
        nums = [w for w in words if re.search(r"\d", w)]

        if len(nums) >= 4 and words[0].isdigit():
            acct_id = words[0]

            # last 4 numeric tokens are TB amounts
            raw_nums = nums[-4:]
            numbers = [clean_number(n) for n in raw_nums]

            if all(n is not None for n in numbers):
                name_tokens = words[1: words.index(raw_nums[0])]
                acct_name = " ".join(name_tokens)

                rows.append([
                    page_no,
                    acct_id,
                    acct_name,
                    numbers[0],
                    numbers[1],
                    numbers[2],
                    numbers[3]
                ])

df = pd.DataFrame(rows, columns=[
    "Page",
    "Account ID",
    "Account Name",
    "Beginning Balance",
    "Debit",
    "Credit",
    "Ending Balance"
])

# df.to_excel(OUTPUT, index=False)

print(f"✅ Extracted rows: {len(df)}")
print(f"✅ Output written to: {OUTPUT}")
df

✅ Extracted rows: 194
✅ Output written to: C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\CSI_Trial_Balance_MAR31_2026.xlsx


,Page,Account ID,Account Name,Beginning Balance,Debit,Credit,Ending Balance
0,1,1007,BANK HNBY/IND OPERATING A 1N0,41131.56,6230355.56,6673729.06,4.977581e+05
1,1,1819,ACCOUNTS. RECEIVABLE MODULE IND,6070798.97,5132179.89,6496233.96,4.796745e+08
2,1,1880,©—=«PREPAID INSURANCE =A IND,0.00,8.09,0.00,8.000000e-02
3,1,1304,FURNITURE & FIXTURES A IND,352.00,638.74,13662.60,8.528700e+04
4,1,1305,"——ACCUMULATIVE DEP FOF A IND -798,",99513.00,12321.82,4340.89,2.910034e+05
...,...,...,...,...,...,...,...
189,14,5907,"INTEREST EXPENSE E IND 395,",13237.00,0.00,0.00,3.951324e+05
190,14,5908,"CORPORATE INCOME TAX E IND 3,",144.06,0.08,8.00,3.144060e+03
191,14,5910,NONDEDUCTIBLE AMEX EXPENSE IND,194289.27,4.00,8000.00,0.000000e+00
192,14,5911,AUTO LEASESE IND,169028.88,0.00,0.60,1.690289e+05


In [48]:
# To read the PDF
import PyPDF2
# To analyze the PDF layout and extract text
from pdfminer.high_level import extract_pages, extract_text
from pdfminer.layout import LTTextContainer, LTChar, LTRect, LTFigure
# To extract text from tables in PDF
import pdfplumber
# To extract the images from the PDFs
from PIL import Image
from pdf2image import convert_from_path
# To perform OCR to extract text from images 
import pytesseract 
# To remove the additional created files
import os

In [49]:
pdf_path =  r"C:\Users\IlaBarshilia\Downloads\PDF Text Parser at OIA\CSI Trial balance at MAR31 2026.pdf"
for pagenum, page in enumerate(extract_pages(pdf_path)):

    # Iterate the elements that composed a page
    for element in page:

        # Check if the element is a text element
        if isinstance(element, LTTextContainer):
            # Function to extract text from the text block
            pass
            # Function to extract text format
            pass

        # Check the elements for images
        if isinstance(element, LTFigure):
            # Function to convert PDF to Image
            pass
            # Function to extract text with OCR
            pass

        # Check the elements for tables
        if isinstance(element, LTRect):
            # Function to extract table
            pass
            # Function to convert table content into a string
            pass